In [2]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

In [3]:
nW = 1024 * 2 * 2 * 2
nG = 1024 * 2 

In [4]:
x0 = 10
def f(x):
    return np.exp((x/x0)**2 - 2)-(x/x0)**2
root = opt.brentq(f, x0, 2*x0) 
a_eff = x0/root
print(x0/root)

0.5637769354091866


In [ ]:
# constants
c = 0.0299792458        # cm/ps
c_si = 2.99792458e8     # m/s
c_cgs = 2.99792458e10   # cm/s

eps_0 = 8.8541878128e-12 # F/m

# Laser pulse duration
tau = 0.150 / 2 / np.sqrt(np.log(2))   # ps

# Laser beam waist size
a0_G = 30e-4 / 2 / np.sqrt(np.log(2))     # поперечный размер cm
a0_D = a_eff*a0_G       # 
# Refractive indices
n_gr = 2.23
n_opt = 2.16

# Other
I_Gauss = 10      # GW/cm^2
m = 1
I_Donut = I_Gauss*np.exp(1)
I_D_si = I_Donut * 1e9 / 1e-4 # W/m^2 = J/s/m^2
I_D_cgs = I_Donut * 1e9 * 1e7 # erg/s/cm^2
I_G_si = I_Gauss * 1e9 / 1e-4 # W/m^2 = J/s/m^2
I_G_cgs = I_Gauss * 1e9 * 1e7 # erg/s/cm^2
# print(f'I\t = {I_si:.2e} (J/s)/m^2 = {I_cgs:.2e} erg/s/cm^2')

d33 = 166   # pm/V
d33_si = d33 * 1e-12 # m/V
d33_cgs = d33 * 3e-8 / (4 * np.pi) # cm/statV
# print(f'd33\t = {d33_si:.2e} m/V = {d33_cgs:.2e} cm/statV')

sqr_E0_D_si  = I_D_si * 2 / (n_opt * c_si * eps_0)      # (V/m)^2
sqr_E0_D_cgs = I_D_cgs * (8 * np.pi / c_cgs / n_opt)    # (statV/cm)^2
sqr_E0_G_si  = I_G_si * 2 / (n_opt * c_si * eps_0)      # (V/m)^2
sqr_E0_G_cgs = I_G_cgs * (8 * np.pi / c_cgs / n_opt)    # (statV/cm)^2
# print(f'E0^2[SI]  = {sqr_E0_si:.2e} (V/m)^2 = {(sqr_E0_cgs * 3e4**2):.2e} (V/m)^2')
# print(f'E0^2[CGS] = {sqr_E0_cgs:.2e} (statV/cm)^2 = {(sqr_E0_si / 3e4**2):.2e} (statV/cm)^2')

py_D_si = eps_0 * d33_si * sqr_E0_D_si  # C  / m^2  = (F/m) * V/m [where C is Coulomb]
py_D_cgs = d33_cgs * sqr_E0_D_cgs       # Fr / cm^2 = statV/cm
py_G_si = eps_0 * d33_si * sqr_E0_G_si  # C  / m^2  = (F/m) * V/m [where C is Coulomb]
py_G_cgs = d33_cgs * sqr_E0_G_cgs       # Fr / cm^2 = statV/cm
print(f'py_D[SI]  = {py_D_si:.2e} C/m^2 = {(py_D_cgs / 3e5):.2e} C/m^2')
print(f'py_D[CGS] = {py_D_cgs:.2e} Fr/cm^2 = {(py_D_si * 3e5):.2e} Fr/cm^2')
print(f'py_G[SI]  = {py_G_si:.2e} C/m^2 = {(py_G_cgs / 3e5):.2e} C/m^2')
print(f'py_G[CGS] = {py_G_cgs:.2e} Fr/cm^2 = {(py_G_si * 3e5):.2e} Fr/cm^2')


In [ ]:
# In CGS units:
print(f'py [kV/cm] = {py_D_cgs * 300 / 1e3:.2e} kV/cm')
# 1 statV/cm = 0.3 kV/cm

# In SI units:
# We need the formulas from the Maxwell equations in SI units
# print(f'py [kV/cm] = {py_si * ???:.2e} kV/cm')

In [ ]:
x = np.linspace(-10, 10, num=1000)
x0 = 2
G = np.exp(-(x/(x0))**2)
Gn = G/np.abs(np.max(G))
f = (x/((a_eff)*x0))**2 * np.exp(-(x/((a_eff)*x0))**2+1)
fn = f/np.abs(np.max(f))
plt.subplots()
plt.plot(x, f)
plt.plot(x, G)
plt.grid(ls=':')
# plt.plot(x, Gn)
# plt.plot(x, fn)
print(1/np.exp(1))

FUNCTIONS

In [ ]:
# epsilon
def eps(w_):
    e_0 = 26
    e_inf = 10
    w_TO = 2 * np.pi * 7.44  # THz
    gamma = 2 * np.pi * 0.844 # THz
    
    epsilon = e_inf + (e_0 - e_inf) * w_TO**2 / (w_TO**2 - w_**2 + 1j * gamma * w_)
    return epsilon

In [ ]:
def k(w_, g_):
    k = np.zeros((np.size(w_), np.size(g_)), dtype=complex)
    
    eps_ = eps(w_)
    
    for i in range(np.size(w_)):
        for j in range(np.size(g_)):
            # k[i][j] = (w_[i] / c)**2 * eps_[i] - g_[j]**2
            if not i == 0:
                k[i][j] = (w_[i] / c) * np.sqrt(eps_[i] - (g_[j] * c / w_[i])**2)
            else:
                k[i][j] = -1j * g_[j]

            if np.imag(k[i][j]) > 0:
                k[i][j] = np.conjugate(k[i][j])
    
    return k 

In [ ]:
def F(w_, tau_):
    return tau_ / (2 * np.sqrt(np.pi)) * np.exp(-(w_ * tau_ / 2)**2)

def G(g_, a0_):
    return -a0_ / (2 * np.sqrt(np.pi)) * np.exp(-(g_ * a0_ / 2)**2)
    # return -a0_ * np.sqrt(np.pi)/ 2 * np.exp(-(g_ * a0_ / 2)**2)*(1 - (g_*a0_)**2/2)
def D(g_, a0_):
    # return a0_ / (2 * np.sqrt(np.pi)) * np.exp(-(g_ * a0_ / 2)**2)
    return -a0_ * np.sqrt(np.pi)/ 2 * np.exp(-(g_ * a0_ / 2)**2)*(1 - (g_*a0_)**2/2)
        

In [ ]:
def ey_field_Gauss(w_, g_, k_):
    ey = np.zeros((np.size(w_), np.size(g_)), dtype=complex)
    FF = F(w_, tau)
    GG = G(g_, a0_G)    
    for i in range(1, np.size(w_)):
        for j in range(1, np.size(g_)):
            arg_up = FF[i] * GG[j]
            arg_down = (k_[i][j] * c / w_[i])**2 - n_gr**2
            ey[i][j] = arg_up / arg_down
    
    return ey
def ey_field_Donut(w_, g_, k_):
    ey = np.zeros((np.size(w_), np.size(g_)), dtype=complex)
    FF = F(w_, tau)
    DD = D(g_, a0_D)    
    for i in range(1, np.size(w_)):
        for j in range(1, np.size(g_)):
            arg_up = FF[i] * DD[j]
            arg_down = (k_[i][j] * c / w_[i])**2 - n_gr**2
            ey[i][j] = arg_up / arg_down
    
    return ey   

CALCULATIONS

In [ ]:
dt = 0.025
w = np.fft.fftfreq(nW, dt/(2 * np.pi))
dw = np.abs(w[1] - w[0])
print(f'w_min = {np.min(w)} rad/s = {np.min(w) / (2 * np.pi)} THz')
print(f'w_max = {np.max(w)} rad/s = {np.max(w) / (2 * np.pi)} THz')
print(f'dw = {dw} rad/s')
print(f'w[0] = {w[0]}')
# plt.plot(w)

In [ ]:
dx = 0.00025
g = np.fft.fftfreq(nG, dx/(2 * np.pi))
dg = np.abs(g[1] - g[0])
print(f'g_min = {np.min(g)} 1/cm')
print(f'g_max = {np.max(g)} 1/cm')
print(f'dg = {dg} 1/cm')
print(f'g[0] = {g[0]}')
# plt.plot(g)

In [ ]:
x = np.fft.fftshift(np.fft.fftfreq(nG, dg/(2 * np.pi)))

In [ ]:
# plt.plot(w/2/np.pi, F(w, tau))

In [ ]:
# plt.plot(g/2/np.pi, F(g, a0))

In [ ]:
print(f'x_max = {np.pi / dg} cm')

print(f'xi_max = {np.pi / dw} ps')

In [ ]:
print(f'dx = {np.pi / np.max(np.abs(g))} cm')
print(f'dt = {np.pi / np.max(np.abs(w))} ps')

In [ ]:
t0_idx = int((0.095 / (c / n_gr)) // dt)
print(f't0_idx = {t0_idx}')

In [ ]:
kk = k(w, g)

In [ ]:
ey_base_f_D = ey_field_Donut(w, g, kk)

In [ ]:
ey_base_f_G = ey_field_Gauss(w, g, kk)

In [ ]:
z = np.linspace(0, 0.1, num=200)

In [ ]:

ey_z_D = np.zeros((np.size(z), np.size(g)), dtype=complex)
for i in range(np.size(z)):
    ey_f_tmp = np.zeros((np.size(w), np.size(g)), dtype=complex)
    
    for j in range(np.size(w)):
        ey_f_tmp[j][:] = ey_base_f_D[j][:] * np.exp(-1j * w[j] * z[i] / (c / n_gr))
        
    ey = np.fft.ifft(np.fft.fft(ey_f_tmp, axis=1), axis=0) * (dg * dw * np.size(w))
    ey_z_D[i][:] = ey[t0_idx][:]

ey_z_D *= (-4 * np.pi * py_D_cgs) * 0.3 # [kV/cm]


In [ ]:

ey_z_G = np.zeros((np.size(z), np.size(g)), dtype=complex)
for i in range(np.size(z)):
    ey_f_tmp = np.zeros((np.size(w), np.size(g)), dtype=complex)
    
    for j in range(np.size(w)):
        ey_f_tmp[j][:] = ey_base_f_G[j][:] * np.exp(-1j * w[j] * z[i] / (c / n_gr))
        
    ey = np.fft.ifft(np.fft.fft(ey_f_tmp, axis=1), axis=0) * (dg * dw * np.size(w))
    ey_z_G[i][:] = ey[t0_idx][:]

ey_z_G *= (-4 * np.pi * py_G_cgs) * 0.3 # [kV/cm]

In [ ]:
ey_t_D = np.zeros(np.size(w), dtype = complex)
ey_f_tmp = np.zeros((np.size(w), np.size(g)), dtype = complex)
for i in range (np.size(w)):
    ey_f_tmp[i][:] = ey_base_f_D[i][:] * np.exp(-1j * w[i] * z[20] / (c / n_gr))
ey = np.fft.ifft(np.fft.fft(ey_f_tmp, axis=1), axis=0) * (dg * dw * np.size(w))
ey_t_D = ey[:][200]
ey_t_D *= (-4 * np.pi * py_D_cgs) * 0.3 # [kV/cm]

In [ ]:
ey_t_G = np.zeros(np.size(w), dtype = complex)
ey_f_tmp = np.zeros((np.size(w), np.size(g)), dtype = complex)
for i in range (np.size(w)):
    ey_f_tmp[i][:] = ey_base_f_G[i][:] * np.exp(-1j * w[i] * z[20] / (c / n_gr))
ey = np.fft.ifft(np.fft.fft(ey_f_tmp, axis=1), axis=0) * (dg * dw * np.size(w))
ey_t_G = ey[:][200]
ey_t_G *= (-4 * np.pi * py_G_cgs) * 0.3 # [kV/cm]

In [ ]:
ey_pt_D = np.zeros(1024*8, dtype = complex)
num_end = int(10//dt-199)
num_init = 52
for i in range(num_init, num_end):
    ey_pt_D[i] = ey_t_D[i].real
print(ey_pt_D[num_end-1])
print(ey_pt_D[num_init+1])

In [ ]:
ey_pt_G = np.zeros(1024*8, dtype = complex)
num_end = int(10//dt-197)
num_init = 52
for i in range(num_init, num_end):
    ey_pt_G[i] = ey_t_G[i].real
print(ey_pt_G[num_end-1])
print(ey_pt_G[num_init+1])

In [ ]:
# osc = np.fft.fftshift(ey_t)
osc_D = ey_pt_D
osc_G = ey_pt_G
plt.figure(dpi=125)
t = np.linspace(0, dt*nW//2, nW//2)
plt.subplots()
plt.plot(t[:500], np.real(osc_D)[:500])
plt.plot(t[:500], np.real(osc_G)[:500])
# plt.plot( np.real(osc)[:])
plt.grid(ls=':')
plt.xlabel('t (ps)')
plt.ylabel('Ey (kV/cm)')
# plt.savefig('Oscillogram_30_EI.pdf',dpi='figure', format = 'pdf') 

In [ ]:

spctr_D =  np.abs(np.fft.fft(ey_pt_D)* dw)
spctr_G =  np.abs(np.fft.fft(ey_pt_G)* dw)
spctr_n_D = spctr_D/np.max(np.abs(spctr_D)) 
spctr_n_G = spctr_G/np.max(np.abs(spctr_G)) 
plt.figure(dpi=125)
plt.subplots()
plt.plot(w[0:nW//4]/2/np.pi, spctr_n_D[0:nW//4])
plt.plot(w[0:nW//4]/2/np.pi, spctr_n_G[0:nW//4])
plt.grid(ls=':')
plt.xlim(0,5)
plt.xlabel('f (Thz)')
plt.ylabel('Spectrum (Arb. Units)')
# plt.savefig('Spectrum_30_EI.pdf',dpi='figure', format = 'pdf') 

In [ ]:
print(f'ey_z_D_min = {np.min(ey_z_D.real):.3f} kV/cm')
print(f'ey_z_D_max = {np.max(ey_z_D.real):.3f} kV/cm')
print(f'ey_z_G_min = {np.min(ey_z_G.real):.3f} kV/cm')
print(f'ey_z_G_max = {np.max(ey_z_G.real):.3f} kV/cm')

In [ ]:
wedge_D = np.transpose(ey_z_D.real)
# wedge = np.transpose(ey_z.imag)
wedge_D = np.fft.fftshift(wedge_D, axes=0)

In [ ]:
plt.figure(dpi=125)
nX = 787
plt.pcolormesh(z*10, x[nX:-nX]*10, wedge_D[nX:-nX, :],
               cmap='bwr',
               vmin=-np.max(np.abs(wedge_D)),
               vmax=np.max(np.abs(wedge_D)),
               )
plt.colorbar(label = 'Ey(kV/cm)')
plt.grid(ls=':')
plt.xlabel('z (mm)')
plt.ylabel('x (mm)')
# plt.savefig('field_Donut_30_EI.pdf',dpi='figure', format = 'pdf') 

In [ ]:
wedge_G = np.transpose(ey_z_G.real)
# wedge = np.transpose(ey_z.imag)
wedge_G = np.fft.fftshift(wedge_G, axes=0)

In [ ]:
plt.figure(dpi=125)
nX = 787
plt.pcolormesh(z*10, x[nX:-nX]*10, wedge_G[nX:-nX, :],
               cmap='bwr',
               vmin=-np.max(np.abs(wedge_G)),
               vmax=np.max(np.abs(wedge_G))
               )
plt.colorbar(label = 'Ey(kV/cm)')
plt.grid(ls=':')
plt.xlabel('z (mm)')
plt.ylabel('x (mm)')
# plt.savefig('field_Gauss_30_EI.pdf',dpi='figure', format = 'pdf') 

In [ ]:
x0 = a0_D
def f(x):
    return np.exp((x/x0)**2 - 2)-(x/x0)**2
root = opt.brentq(f, x0, 2*x0) 
root2 = opt.brentq(f, 0, x0)
print(root)
print(root2)
a_ch = (root-root2)/2
print(a_ch)

In [ ]:
 e_0 = 26
def omega(t, a):
    e_0 = 26
    e_inf = 10
    w_TO = 2 * np.pi * 7.44  # THz
    gamma = 2 * np.pi * 0.844 # THz
    t_eff = np.sqrt(t**2 + (e_0 - n_gr**2)*(a/c)**2)
    t_EFF = np.sqrt(t_eff**2 + 2*gamma*(e_0 - e_inf)*0.05/(w_TO**2*c*np.sqrt(e_0 - n_gr**2))) 
    # w = 1/(np.pi*np.sqrt(2)*t_EFF)
    # w = 1/(np.pi*np.sqrt(2)*t_eff)
    w = 1/(t_eff)
    return w 
nu_0 = omega(tau, a0_G)   # синяя
nu_1 = omega(tau, a_ch)   # желтая
# nu_2 = c/(2*2*(a_ch)*np.sqrt(e_0)*np.sin(64*np.pi/180))    # красная
nu_2 = c/(2*2*(a_ch)*5.38*np.sin(64*np.pi/180))    # красная

print(nu_0)
print(nu_1)
print(nu_2)

In [ ]:
plt.figure(dpi=125)
plt.subplots()
plt.plot(w[0:nW//4]/2/np.pi, spctr_n_D[0:nW//4])
plt.plot(w[0:nW//4]/2/np.pi, spctr_n_G[0:nW//4])
plt.axvline(nu_0)
plt.axvline(nu_2, color = 'red')
plt.axvline(nu_1, color = 'yellow')
plt.grid(ls=':')
plt.xlim(0,6)
plt.xlabel('f , Thz')
plt.ylabel('Spectrum Arb. Units')
# plt.savefig('Spectrum_30_EI_vlines.pdf',dpi='figure', format = 'pdf')

In [ ]:
l = tau*c/np.sqrt(26-n_gr**2)
print(l)